In [1]:
import os
import shutil
import matplotlib.pyplot as plt

import torch as t
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

In [2]:
DEVICE = 'cuda' if t.cuda.is_available() else 'cpu'
print(f"Available device = {DEVICE}")

Available device = cuda


In [3]:
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

IMAGE_SIZE = 224
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.08, 1.0), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([transforms.RandomRotation(degrees=30)], p=0.3),
    transforms.RandomApply([transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1)], p=0.8),
    transforms.RandomGrayscale(p=0.2),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), 
                std=(0.229, 0.224, 0.225)),
])

In [4]:
train_dataset = datasets.Food101(root="./data", split='train', download=False, transform=train_transform)
val_dataset = datasets.Food101(root="./data", split='test', download=False, transform=val_transform)

In [5]:
BATCH_SIZE = 32*4

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train length = {len(train_loader)*BATCH_SIZE}, Test length = {len(val_loader)*BATCH_SIZE}")

Train length = 75776, Test length = 25344


In [6]:
from config import Config
from swin import swin_transformer
import train

In [7]:
epochs = 100
lr = 1e-4
num_classes = 211

In [8]:
swin_model = swin_transformer(in_channels=Config.in_channels, hidden_dim=Config.hidden_dim, layers=Config.layers, 
                        downscaling_factor=Config.downscaling_factor, heads=Config.heads, 
                        head_dim=Config.head_dim, window_size=Config.window_size, 
                        relative_position=Config.relative_pos, num_clas=num_classes).to(DEVICE)

In [9]:
# model_path = "F:\\CBIR\\SWIN\\runs1\\mdl-78.pt"
# swin_model.load_state_dict(t.load(model_path, map_location=t.device(DEVICE), weights_only=True), strict=False)

In [10]:
loss = t.nn.CrossEntropyLoss()
optimizer = t.optim.AdamW(swin_model.parameters(), lr=lr)

In [11]:
training = train.Trainer(train_loader, val_loader, epochs, loss, optimizer, DEVICE)
history = training.start(swin_model)

Test batch processing: 100%|████████████████████████████████████████| 198/198 [05:08<00:00,  1.56s/Batchs, loss=5.9314]


1/100 | train loss = 4.5443 | test loss = 5.7041


Test batch processing: 100%|████████████████████████████████████████| 198/198 [03:34<00:00,  1.08s/Batchs, loss=5.4123]


2/100 | train loss = 4.2618 | test loss = 5.1681


Test batch processing: 100%|████████████████████████████████████████| 198/198 [03:34<00:00,  1.09s/Batchs, loss=3.5554]


...

26/100 | train loss = 2.2982 | test loss = 3.8565


Test batch processing:  51%|████████████████████▍                   | 101/198 [01:52<01:48,  1.12s/Batchs, loss=4.3396]IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

Test batch processing: 100%|████████████████████████████████████████| 198/198 [03:45<00:00,  1.14s/Batchs, loss=3.3280]


30/100 | train loss = 2.1537 | test loss = 3.5997


Batch processing:  53%|███████████████████████▊                     | 313/592 [06:04<05:23,  1.16s/Batchs, loss=2.3968]

KeyboardInterrupt: 